# Week 1 Day 2 Lab - Introducing UnderCover

## LLMs and Agents on a commercial task: insurance claims

A claims handler at UnderCover Travel Insurance has already decided each of the 25 claims in its queue. We send every claim to a few models, along with the customer's policy record and the policy wording. Each model replies with a decision of approve, decline or refer, a rationale and a payout. Then we compare the answers with the handlers' decisions and look at what each model cost.

At the end we give the same job to a LangChain agent that uses tools to look up the policy and record its decision.

## 4 parts:

PART 1: Examine the data

PART 2: Run with some models

PART 3: Optional: larger models and Jev

PART 4: An Agent via LangChain

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This lab calls paid models through OpenRouter and costs only a few cents. If you're OK to spend that, you'll need some credit on your account at <a href="https://openrouter.ai">openrouter.ai</a>. If you'd rather not spend anything, there's a free version of this lab in <a href="week1day2free.ipynb">week1day2free.ipynb</a>.<br/><br/>
            Please execute each cell and check that you get similar results to mine (the models vary a little from run to run). Add print statements and try small adjustments to make the code your own.<br/><br/>
            Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Resources</h2>
            <span style="color:#118AB2;">The <a href="../guides/01_intro.ipynb">guides</a> folder in this repo has more information on topics like the command line, Git, Python and debugging.<br/><br/>
            If this lab is too much, consider courses 1, 2, 4 and 5 in <a href="https://edwarddonner.com/curriculum">my curriculum</a>: AI Builder, AI Coder, and the Core and Agentic Tracks of AI Engineer.<br/><br/>
            If it's too slow, put me on 2x and zip through it! There's some interesting content on Jev for the pros.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/connect.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Stay in touch</h2>
            <span style="color:#118AB2;">Please message me in Udemy anytime with questions, or email me at <a href="mailto:ed@edwarddonner.com">ed@edwarddonner.com</a>.<br/><br/>
            Let's connect on <a href="https://www.linkedin.com/in/eddonner/">LinkedIn</a>! And subscribe to my <a href="https://www.youtube.com/@edward.donner">YouTube channel</a> for more on LLMs, agents and coding agents.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

## Connect to OpenRouter

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

OpenRouter API key found


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: the Chat Completions API</h2>
            <span style="color:#118AB2;">We call models with the Chat Completions API. <code>messages</code> is a Python list of dicts, each with a <code>role</code> and <code>content</code>. The system prompt tells the model how to behave and the user prompt has the request. The reply is in <code>response.choices[0].message.content</code>.<br/><br/>
            Here's a quick test.
            </span>
        </td>
    </tr>
</table>

In [3]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"}
]

response = openrouter.chat.completions.create(model="openai/gpt-6-luna", messages=messages)
display(Markdown(response.choices[0].message.content))

Arrr, there’s no universally agreed answer. The universe may have no built-in purpose—or its purpose may be beyond our ken. But meaning can still be real: we make it through love, curiosity, kindness, and the lives we touch. Perhaps the grand voyage is not finding a single hidden treasure, but choosing what’s worth sailing for.

## Part 1: Load and examine the data

The seed data is in the repo's `data` folder. `get_sample_data` returns the claims and the handlers' decisions on them, keyed by claim id. The model is sent the claims, the policies and the policy wording.

By default we work with the queue of 25 claims. If you are using free models with a daily limit, set `TINY = True` to keep 5 of them.

In [4]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [5]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

25 claims, 25 labels, 50 policies
The policy wording is 1,369 words


In [6]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

Here is the handler's decision on that claim which we'll use to validate the LLM results.

In [7]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

### Explore the data

It's always good to dig into the data before running experiments.

In [8]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,claim_type,amount_claimed,difficulty,decision,payout
0,CLM-1001,"Flight home canceled by the airline, rebooked ...",trip_delay,420.0,routine,approve,420.0
1,CLM-1002,Suitcase lost on a connecting flight,baggage,1150.0,routine,approve,580.0
2,CLM-1003,"Hospital visit abroad after a fall, fractured ...",emergency_medical,3800.0,routine,approve,3750.0
3,CLM-1004,Checked bag arrived 31 hours late,baggage_delay,186.0,routine,approve,186.0
4,CLM-1005,Passport eaten by a giraffe at breakfast,passport,243.0,routine,approve,243.0
5,CLM-1006,Hit by a falling coconut while reading under a...,emergency_medical,425.0,routine,approve,425.0
6,CLM-1007,Phone dropped in the hotel shower,baggage,1099.0,routine,approve,50.0
7,CLM-1008,Trip to Seattle canceled after a tornado carri...,trip_cancellation,2440.0,routine,approve,2240.0
8,CLM-1009,Camera snatched on the Barcelona metro,baggage,980.0,routine,approve,930.0
9,CLM-1010,Emergency dental treatment in Las Vegas for a ...,emergency_medical,1350.0,routine,decline,0.0


Claim CLM-1015 might be a familiar example...

In [9]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


Here's another one that might ring a bell...

In [10]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


This chart counts the claims of each type, split by the handler's decision.

In [11]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

Here's a breakdown by difficulty. The policy wording is enough to decide the routine and hard claims. The manual claims depend on the claims manual or the customer's history.

In [12]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

## Part 2: running with some LLMs

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: Structured Outputs</h2>
            <span style="color:#118AB2;">With Structured Outputs the model replies in JSON that follows a schema we give it. We describe the schema as a Pydantic class and pass it to <code>parse()</code> as the <code>response_format</code>. The reply comes back as an object of that class in <code>response.choices[0].message.parsed</code>.
            </span>
        </td>
    </tr>
</table>

Starting by defining a Pydantic class to describe the output so we can use Structured Outputs.

The rationale comes before the decision to ensure the model thinks through the decision before making it.

In [13]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## The prompt

The system prompt has the instructions and the whole policy wording. The user prompt has the customer's policy record and the claim, both as JSON.

In [14]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

### Calling the LLM

In [15]:
def decide(claim, model, effort):
    """Ask one model to decide one claim."""
    start = time.time()
    response = openrouter.chat.completions.parse(
        model=model,
        messages=make_messages(claim),
        response_format=ClaimDecision,
        reasoning_effort=effort,
        extra_body={"provider": {"require_parameters": True}},
    )
    answer = response.choices[0].message.parsed
    return {
        "model": model,
        "effort": effort,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 1),
    }

In [16]:
first = decide(claims[0], "openai/gpt-6-luna", "low")
first

{'model': 'openai/gpt-6-luna',
 'effort': 'low',
 'technique': 'LLM',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': 'The cancellation caused a delay from the scheduled departure on June 18 until the actual departure on June 19, exceeding the Premium plan’s 6-hour waiting period, and crew problems and canceled service are covered under section 3. The attached carrier notice confirms the cause and departure times, and receipts support the reasonable accommodation, meal and transport expenses; the $420 eligible total is within the $800 maximum, with no deductible under section 10.',
 'cost': 0.0003998,
 'seconds': 2.2}

### Now running multiple models

In [17]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, effort, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [18]:
luna_low = run("openai/gpt-6-luna", "low", claims, labels)

openai/gpt-6-luna at low: 17 of 25 right for $0.0043 in 53 seconds


In [19]:
luna_medium = run("openai/gpt-6-luna", "medium", claims, labels)

openai/gpt-6-luna at medium: 18 of 25 right for $0.0062 in 82 seconds


In [20]:
luna_high = run("openai/gpt-6-luna", "high", claims, labels)

openai/gpt-6-luna at high: 18 of 25 right for $0.0070 in 100 seconds


In [21]:
gemini = run("google/gemini-3.5-flash-lite", "low", claims, labels)

google/gemini-3.5-flash-lite at low: 16 of 25 right for $0.0328 in 29 seconds


## Results by difficulty

This counts the claims each entry got right at each level of difficulty. Expect nearly all of the manual claims to be wrong for now.

In [22]:
scored = pd.DataFrame(luna_low + luna_medium + gemini)
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)

difficulty                           routine  hard  manual
model                        effort                       
openai/gpt-6-luna            low          13     2       2
                             medium       14     4       0
google/gemini-3.5-flash-lite low          14     1       1

## Where a model got it wrong

For each claim a model got wrong, this shows its rationale, the handler's reason and what the claim is there to test. Here we look at Luna at `low`. Pass in another set of results to see its misses.

In [23]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(luna_low, labels)

**CLM-1002, routine**: refer $0.00, where the handler said approve $580.00

- Model: The loss is covered under Section 5, and the airline’s lost-baggage report and settlement letter document the carrier loss and its $400 payment. However, receipts are missing for the toiletry kit and shirts, so the required documentation is incomplete; under Section 11, the claim must wait for the missing documents. The claim was submitted within 90 days of the incident.
- Handler: The loss was reported to the carrier and the carrier has declared the bag lost. The suit is over the $300 per-item limit on Premium, so it counts at $300. The carrier's $400 comes off, then the $50 deductible.
- Tests: Payout arithmetic: the per-item limit, the carrier's payment and the deductible in order


**CLM-1010, routine**: refer $0.00, where the handler said decline $0.00

- Model: Refer for review under sections 7 and 9. The clinic records describe a recent extraction that the claimant believes he performed himself, and document heavy drinking and signs of impairment, but do not establish whether he was impaired when the tooth was removed; the alcohol exclusion therefore cannot be conclusively applied from the available evidence.
- Handler: The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.
- Tests: A general exclusion: a loss while impaired by alcohol, found in the clinical notes


**CLM-1019, manual**: approve $210.00, where the handler said refer $0.00

- Model: The carrier confirms a nine-hour delay caused by a crew duty-time limit, which meets the Premium plan’s six-hour waiting period and the crew-problem requirement in section 3. The attached confirmation and receipts support the reasonable extra costs claimed; section 10 allows the full eligible amount, subject to the $800 Trip Delay maximum, with no deductible for this benefit.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The theft occurred during the covered trip, and the police report documents it on the same day, satisfying section 5’s theft-report requirement. Eligible items total $1,103; Electronics Plus raises the per-item limit for electronics to $1,200, and the Premium baggage maximum is $1,500. After the $50 deductible, the payout is $1,053; the tote’s value is stated but not separately evidenced, so it is not included in the eligible amount.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


**CLM-1022, manual**: approve $85.00, where the handler said approve $155.00

- Model: The carrier confirms a severe-weather delay of more than three hours, so the claim qualifies under the First plan’s Trip Delay benefit in section 3. The $85 rebooking fee is supported by a receipt and is an eligible expense; the $140 meal claim has no required receipts, so it cannot be included. The payout is below the $1,500 maximum, and no deductible applies under section 10.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1023, hard**: approve $61.00, where the handler said refer $0.00

- Model: The delay exceeded the First plan’s 3-hour waiting period, and a mechanical fault is a covered cause under section 3. The receipts support $61 in reasonable meal costs; the carrier certificate confirms the delay and timing, and no deductible applies to Trip Delay.
- Handler: The delay of 9 hours 17 minutes is well past the 3 hour waiting period on First and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the fuel pump fault rests on the customer's account alone. Document request: Northwind's confirmation of the reason for the delay. The claim appears payable at $61 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


**CLM-1024, manual**: approve $250.00, where the handler said refer $0.00

- Model: The phone was stolen during the covered trip and reported to local police the same day, with the police report and purchase receipt attached; no exclusion appears to apply. Under section 5, the Premium per-item limit is $300 and Electronics Plus was not purchased, so section 10 caps the eligible amount at $300; the $50 deductible applies and no other payment is reported.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


**CLM-1025, hard**: approve $71.00, where the handler said refer $0.00

- Model: Approve under Section 3: the carrier certificate confirms the flight was delayed from 11:55 a.m. until 2:10 a.m. the next day because the first officer exceeded duty hours and a replacement was needed, a crew problem. The delay exceeds Economy’s 12-hour waiting period, and the attached receipts support $71 in meal costs; this is below the $300 Trip Delay maximum, and no deductible applies.
- Handler: The delay of 14 hours 15 minutes is past the 12 hour waiting period on Economy and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the crew problem rests on the customer's account alone. Document request: Skylark's confirmation of the reason for the delay. The claim appears payable at $71 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


### What it cost

This shows how many claims each entry got right, what it cost and how long it took for the whole queue.

In [24]:
scored.groupby(["model", "effort"], sort=False)[["right", "cost", "seconds"]].sum()

right      cost  seconds
model                        effort                          
openai/gpt-6-luna            low        17  0.004349     52.9
                             medium     18  0.006163     82.5
google/gemini-3.5-flash-lite low        16  0.032773     28.6

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Accuracy, cost and speed are the numbers a customer will ask about. Cost per claim is the easiest one to relate to the business: compare it with what a handler's time on a claim costs.<br/><br/>
            The kind of mistake matters too. A wrong approval pays out money the policy doesn't owe, a wrong decline lets a customer down and a wrong referral costs a handler's time.
            </span>
        </td>
    </tr>
</table>

## Part 3: Optional extras

These two experiments on the queue are optional. Nothing later in the lab depends on them.

### Sol at high

GPT-6 Sol is Luna's bigger sibling; we run it at the `high` reasoning level to see how much better it does on this prompt.

In [25]:
sol = run("openai/gpt-6-sol", "high", claims, labels)

openai/gpt-6-sol at high: 19 of 25 right for $0.1201 in 88 seconds


### Jev

Jev, from TypeSafe AI, doesn't generate text. We send it the facts as `state` along with a typed question, and it returns a probability for each option and a confidence. It chooses between approve, decline and refer in a fraction of a second. It can't write a rationale or work out a payout, so we score it on the decision alone.

Jev is on OpenRouter's Decisions endpoint, which the OpenAI client can't reach. We use OpenRouter's own Python client with the same key.

In [26]:
JEV = "typesafe/jev-1.13"
JEV_QUESTION = {
    "type": "choice",
    "instructions": "You are a claims handler at UnderCover Travel Insurance. Decide `claim` using `policy_wording` and the customer's `policy` record.",
    "criteria": {
        "approve": "The wording covers the claim and the required documents are attached.",
        "decline": "The wording does not cover the claim.",
        "refer": "A colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.",
    },
}

openrouter_decisions = OpenRouter(api_key=openrouter_api_key, timeout_ms=60_000)


def decide_with_jev(claim):
    """Ask Jev for the decision on one claim, with its confidence."""
    state = {"policy_wording": policy_wording, "policy": policies[claim["policy_id"]], "claim": claim}
    start = time.time()
    response = openrouter_decisions.alpha.decisions.create(model=JEV, state=state, questions={"decision": JEV_QUESTION})
    answer = response.answers["decision"]
    return {
        "claim_id": claim["claim_id"],
        "decision": answer.choice,
        "confidence": answer.confidence,
        "right": answer.choice == labels[claim["claim_id"]]["decision"],
        "cost": response.usage.cost,
        "seconds": round(time.time() - start, 2),
    }

In [27]:
jev = pd.DataFrame([decide_with_jev(claim) for claim in claims])
luna_decisions_right = sum(result["decision"] == labels[result["claim_id"]]["decision"] for result in luna_low)

print(f"Jev: {jev['right'].sum()} of {len(jev)} decisions right for ${jev['cost'].sum():.4f} in {jev['seconds'].sum():.0f} seconds")
print(f"Luna at low: {luna_decisions_right} of {len(luna_low)} decisions right")

Jev: 16 of 25 decisions right for $0.0033 in 6 seconds
Luna at low: 19 of 25 decisions right


Here are Jev's decisions split at a confidence of 0.9. One way to use a fast model like this is to keep the answers it's confident about and send the rest to a stronger model.

In [28]:
jev["confident"] = jev["confidence"] >= 0.9
jev.groupby("confident")["right"].agg(["sum", "count"])

,sum,count
confident,,
False,12,20
True,4,5


## Part 4: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Reminder: LangChain create_agent</h2>
            <span style="color:#118AB2;">1. Describe the LLM with a LangChain class like <code>ChatOpenAI</code><br/>
            2. Turn a function into a tool with <code>@tool</code><br/>
            3. <code>create_agent</code> builds the agent and <code>invoke()</code> runs its loop<br/><br/>
            Behind the scenes, a LangGraph graph is created; we'll find out more in Week 2.
            </span>
        </td>
    </tr>
</table>

In [30]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

Ahoy there, ye curious landlubber! Let me spin ye a tale o' the cosmos, for the meaning o' the universe be a question as deep as the Mariana Trench!

**From the Decks of Science:**
The universe began with a mighty *Big Bang*—a cosmic explosion that set all things ablaze like a cannonball in a stormy sky. From that fiery heart, galaxies formed, stars ignited, and planets rose like barnacles on a ship's hull. We're just stardust given form, sailing through this grand void. The laws of physics are the compass by which we navigate this grand voyage—gravity, light, time itself—they chart the course of our existence.

**From the Heart of the Sea:**
But what of *purpose*? Ah, that be where the real treasure lies! Some say the universe was created by a Great Captain—an Almighty or an ancient god—who set out to explore and discover. Others believe we are the universe becoming aware of itself, like a ship discovering its own map. Perhaps the meaning be found in the journey itself: to seek knowledge, to love one another, to brave the unknown, and to leave behind a legacy that echoes through eternity.

**In Pirate Terms:**
So tell me, matey—what be yer true quest? Are ye lookin' fer gold and glory across the seven seas? Or do ye seek the deeper truth of why we sail at all? The universe may be vast and mysterious, but every soul aboard it carries a piece of the mystery within. That's where the real treasure be—not in the stars themselves, but in the hearts that gaze upon them and dare to dream.

Now go forth and find yer own meaning, ye scallywag! 🏴‍☠️⚓️

In [31]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses Luna at `low` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with Luna at `low`.

In [32]:
luna = ChatOpenAI(model="openai/gpt-6-luna", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, reasoning_effort="low", timeout=120, max_retries=5)

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=luna, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [33]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call_ulpU7d8CAu66UfJZHAVsVVqx)
 Call ID: call_ulpU7d8CAu66UfJZHAVsVVqx
  Args:
    policy_id: POL-2001
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "plan": "premium",
  "add_ons": [],
  "purchased_on": "2026-04-22",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Rome, Italy",
    "departs_on": "2026-06-10",
    "returns_on": "2026-06-18",
    "trip_cost": 3100.0
  },
  "travelers": [
    "Maya Patel"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call_ONUCvhOqsmuB4KcqgyPLQKOF)
 Call ID: call_ONUCvhOqsmuB4KcqgyPLQKOF
  Args:
    claim_id: CLM-1001
    decision: approve
    payout: 420
    rationale: The airline confirmation documents a cancellation ca

Now the whole queue. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. OpenRouter's cost for each call is in the message metadata. A claim with no recorded decision counts as wrong.

In [34]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    cost = sum(message.response_metadata["token_usage"]["cost"] for message in result["messages"] if message.type == "ai")
    return {
        "model": "openai/gpt-6-luna",
        "effort": "low",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "cost": cost,
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

25 of 25 claims have a recorded decision


In [35]:
pd.DataFrame(luna_low + agent_results).groupby(["model", "effort", "technique"], sort=False)[["right", "cost", "seconds"]].sum()

right      cost  seconds
model             effort technique                          
openai/gpt-6-luna low    LLM           17  0.004349     52.9
                         Agent         18  0.008579    142.5

In [36]:
show_misses(agent_results, labels)

**CLM-1004, routine**: approve $74.00, where the handler said approve $186.00

- Model: Section 4 covers essential clothing and toiletries bought while waiting for checked baggage delivered after the Premium plan’s 12-hour threshold; the carrier report shows delivery 31 hours and 15 minutes after arrival, and the receipts are attached. I allow the $27 underwear and $47 basic toiletries ($74 total); the $112 dress was purchased for a welcome dinner rather than as essential clothing, so it is not eligible under section 4.
- Handler: The carrier's report shows the bag arrived 31 hours after the passenger, past the 12 hour waiting period on Premium. The purchases are essential clothing and toiletries with receipts and are inside the $300 maximum. No deductible applies to Baggage Delay.
- Tests: A routine baggage delay, with no deductible on this benefit


**CLM-1018, manual**: approve $330.00, where the handler said refer $0.00

- Model: The theft occurred during the covered trip, and the attached Carabinieri report documents both claimed items and supports the required prompt police report under section 5. Under sections 1, 5 and 10, the Premium per-item limit reduces the jacket’s eligible amount to $300; adding the $80 daypack and subtracting the $50 baggage deductible gives a payout of $330.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019, manual**: approve $210.00, where the handler said refer $0.00

- Model: The Premium plan's 6-hour waiting period is met: the documented delay was 9 hours 20 minutes and was caused by a crew duty-time problem, covered under section 3. The attached carrier confirmation and receipts support $210 in reasonable accommodation, meal and transport expenses; this is within the $800 Trip Delay maximum, and no deductible applies.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: The claim describes theft/loss of belongings during the covered trip, but section 5 requires theft to be reported to local police within 24 hours and a copy of the police report provided. The attached tour-guide letter is not a police report, so a required document is missing; refer for follow-up rather than determine payment now.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The theft occurred during the covered trip, and the police report confirms the snatching and was made the same day, meeting section 5's police-report requirement. The eligible items total $1,103; the Electronics Plus limit covers each listed electronic item in full, the tote is below the standard per-item limit, and after the $50 Premium deductible under section 10 the payout is $1,053, within the baggage maximum.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


**CLM-1022, manual**: refer $0.00, where the handler said approve $155.00

- Model: The carrier report confirms a 30-hour delay caused by severe weather, exceeding the First plan’s 3-hour waiting period under section 3. The claimed $140 meal expense has no receipts, which section 3 requires; please obtain them before deciding the eligible expenses, including whether the $85 rebooking fee was a reasonable cost during the delay.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1024, manual**: approve $250.00, where the handler said refer $0.00

- Model: The phone was stolen during the covered trip and was in the customer's hand, not unattended; the police report was made the same day as required by section 5, and the purchase receipt supports the item. Under section 10, the eligible amount is capped at the Premium per-item limit of $300, then reduced by the $50 baggage deductible, for a payout of $250.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


## Putting it all together

Here is every experiment in one table. Cost is for the whole queue and speed is the average time for one claim. Jev isn't included because it only makes the decision.

In [37]:
# If you skipped Part 3, take sol out of this list
experiments = [luna_low, luna_medium, gemini, sol, agent_results]

rows = []
for results in experiments:
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "reasoning": results[0]["effort"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "cost": f"${cost:.4f}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,reasoning,technique,accuracy,cost,speed
0,openai/gpt-6-luna,low,LLM,17 / 25,$0.0043,2.1 seconds a claim
1,openai/gpt-6-luna,medium,LLM,18 / 25,$0.0062,3.3 seconds a claim
2,google/gemini-3.5-flash-lite,low,LLM,16 / 25,$0.0328,1.1 seconds a claim
3,openai/gpt-6-sol,high,LLM,19 / 25,$0.1201,3.5 seconds a claim
4,openai/gpt-6-luna,low,Agent,18 / 25,$0.0086,5.7 seconds a claim


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">This lab is about experimenting with different LLMs and Agents to understand the accuracy, cost and speed.<br/><br/>
            Based on our initial experiments, Luna (low) is able to perform well at a low cost. More expensive models can perform better.<br/><br/>
            Challenge: with a goal of at least 15 / 25 correct consistently, using the agent, what is the cheapest model you can find?<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#069D75;">Congratulations! And if I could disturb you with a quick request:</h2>
            <span style="color:#069D75;">You've finished the first main lab. You compared several models and an agent on UnderCover's claims, with accuracy, cost and speed side by side.<br/><br/>
            On Day 3 we bring in coding agents and start building the UnderCover platform.<br/><br/>
            <b>I have a quick request for you. If you have a moment to rate the course, I'd hugely appreciate it. It's one of the main ways Udemy decides whether
            to show the course to others, and I'd be super grateful.</b>
            </span>
        </td>
    </tr>
</table>